# 03 · SFT a native-schema coding agent

Train a language-only LoRA on validated, replayable trajectories.
The notebook defaults to a two-step plumbing smoke test; the real
run remains gated until the baseline, schema-survival, split, and
replay checks pass.

This stage is agentic-coding-only by policy: the corpus contains
no general-capability replay slice, and drift on non-coding chat
is an accepted trade rather than a gate. Specialisation comes
from concentrated coding gradients and adapter capacity — never
from unlearning objectives on general text, which regress the
coding gates themselves. Acceptance still requires coding,
tool-protocol and harness-safety non-regression against the
frozen baseline.

**Input:** a private dataset from notebook 02.
**Output:** a versioned LoRA adapter and, at the end of a full run,
the merged SFT checkpoint that notebook 04 starts from.

## Install the pinned day-zero environment

In [ ]:
import subprocess
import sys
from pathlib import Path

# The Git pins supply current Unsloth/Qwen3.8 support. Transformers, TRL and
# Datasets deliberately use the mutually compatible versions from the adjacent
# official Unsloth Qwen3.5 27B notebook. Do not replace these with branch-head
# SHAs without resolving package metadata together first.
GIT_REVISIONS = {
    "unsloth": "c87fe20e32aca9ceb2dc5059c2987738f32446e8",
    "unsloth_zoo": "5b239e574f03ab3077c17e49aeef3cacfe7cdd4e",
}

import torch

torch_version = torch.__version__.split("+", 1)[0]
torch_minor = ".".join(torch_version.split(".")[:2])
torchao_by_torch = {"2.8": "0.16.0", "2.9": "0.16.0", "2.10": "0.16.0", "2.11": "0.18.0"}
xformers_by_torch = {"2.8": "0.0.32.post2", "2.9": "0.0.33.post1", "2.10": "0.0.34", "2.11": "0.0.34"}
if torch_minor not in torchao_by_torch:
    raise RuntimeError(
        f"No reviewed Colab dependency set for torch {torch.__version__}. "
        f"Expected one of {sorted(torchao_by_torch)}; update the compatibility matrix first."
    )

COMPATIBILITY_PINS = {
    "transformers": "5.3.0",
    "trl": "0.22.2",
    "datasets": "4.3.0",
    "peft": "0.19.0",
    "torchao": torchao_by_torch[torch_minor],
    "xformers": xformers_by_torch[torch_minor],
}
INSTALLER_REVISION = "colab-v2"
pin_key = "-".join(value.replace(".", "") for value in COMPATIBILITY_PINS.values())
git_key = "-".join(value[:8] for value in GIT_REVISIONS.values())
INSTALL_KEY = f"{INSTALLER_REVISION}-torch{torch_minor}-{git_key}-{pin_key}"
INSTALL_MARKER = Path(f"/content/.qwen38_env_{INSTALL_KEY}")
PIP_LOG = Path("/content/qwen38_pip_install.log")
FORCE_INSTALL = False

def install_phase(name: str, packages: list[str], *, no_deps: bool = False) -> None:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "--no-cache-dir",
        "--log",
        str(PIP_LOG),
    ]
    if no_deps:
        command.append("--no-deps")
    command.extend(packages)
    print(f"\n=== install phase: {name} ===")
    print("\n".join(f"  {package}" for package in packages))
    result = subprocess.run(command, check=False)
    if result.returncode:
        log_tail = (
            "\n".join(PIP_LOG.read_text(errors="replace").splitlines()[-120:])
            if PIP_LOG.exists()
            else "[pip did not create its log file]"
        )
        print(f"\n--- tail of {PIP_LOG} ---\n{log_tail}")
        raise RuntimeError(
            f"Package installation failed during {name!r} with exit code {result.returncode}. "
            f"The detailed log is at {PIP_LOG}."
        )

if FORCE_INSTALL or not INSTALL_MARKER.exists():
    if PIP_LOG.exists():
        PIP_LOG.unlink()
    install_phase("packaging tools", ["pip", "setuptools==80.9.0", "wheel>=0.42.0"])
    install_phase("Qwen3.8 training stack", [
        f"unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git@{GIT_REVISIONS['unsloth_zoo']}",
        f"unsloth @ git+https://github.com/unslothai/unsloth.git@{GIT_REVISIONS['unsloth']}",
        f"torch=={torch_version}",
        f"torchao=={COMPATIBILITY_PINS['torchao']}",
        f"transformers=={COMPATIBILITY_PINS['transformers']}",
        f"trl=={COMPATIBILITY_PINS['trl']}",
        f"datasets=={COMPATIBILITY_PINS['datasets']}",
        f"peft=={COMPATIBILITY_PINS['peft']}",
        "accelerate",
        "bitsandbytes",
        "trackio",
        "huggingface_hub>=0.34.0,<2.0",
        "hf_transfer",
        "sentencepiece>=0.2.0",
        "protobuf",
        "pytest",
        "jmespath",
    ])
    install_phase(
        "PyTorch-matched xFormers wheel",
        [f"xformers=={COMPATIBILITY_PINS['xformers']}"],
        no_deps=True,
    )
    INSTALL_MARKER.write_text(INSTALL_KEY)
    print("Packages installed. Restart the Colab runtime, then rerun this notebook from the top.")
else:
    print(f"Pinned environment already installed: {INSTALL_KEY}")

After the first install, restart the runtime and rerun the notebook from the top; the install marker skips the pip work.

In [ ]:
import gc
import json
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import login, whoami

if "GIT_REVISIONS" not in globals():
    raise RuntimeError(
        "This runtime was restarted. Rerun the notebook from the first cell; "
        "the install marker will skip the expensive package installation."
    )
if "COMPATIBILITY_PINS" not in globals():
    raise RuntimeError("Missing compatibility pins; rerun the notebook from the first cell.")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab G4 GPU runtime before continuing.")

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3
# A vendor-labelled 96 GB card can be reported as about 89.4 GiB because
# PyTorch converts the byte count with a binary divisor. Keep the floor well
# above the roughly 44.7 GiB reported for a 48 GB card without rejecting G4.
MIN_G4_TOTAL_GIB = 85.0
print(
    f"GPU: {gpu.name} ({gpu_total_gib:.1f} GiB total), "
    f"capability={torch.cuda.get_device_capability(0)}"
)
if gpu_total_gib < MIN_G4_TOTAL_GIB:
    raise RuntimeError(
        "This suite expects the nominal 96 GB Colab G4 runtime. "
        f"PyTorch reports {gpu_total_gib:.1f} GiB total; expected at least "
        f"{MIN_G4_TOTAL_GIB:.0f} GiB. A value near 45 GiB usually indicates "
        "the 48 GB GPU variant."
    )

# IPython stores the last exception on sys.last_traceback, whose frames keep
# every local alive, including a ~52 GiB model from a failed cell. gc.collect()
# cannot free what those frames still reference.
def release_stale_gpu_state() -> float:
    for _stale_name in ("model", "tokenizer", "processor", "trainer"):
        globals().pop(_stale_name, None)
    for _exc_attr in ("last_traceback", "last_value", "last_type", "last_exc"):
        if hasattr(sys, _exc_attr):
            delattr(sys, _exc_attr)
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch._dynamo.reset()
    except AttributeError:
        pass
    return torch.cuda.mem_get_info()[0] / 1024**3

# Fail before a model load that accelerate would silently offload.
def require_free_vram(minimum_gib: float) -> float:
    free_gib = release_stale_gpu_state()
    if free_gib < minimum_gib:
        raise RuntimeError(
            f"Only {free_gib:.1f} GiB VRAM is free but this load needs about "
            f"{minimum_gib:.0f} GiB. A previous model in this kernel is still "
            "holding memory. Restart the runtime and rerun from the top."
        )
    return free_gib

# Reject a load that accelerate quietly spilled to CPU or disk. A partially
# offloaded model copies weights back per forward pass (the 2.4 GiB embedding
# alone) and is guaranteed to OOM or crawl mid-episode.
def assert_model_fully_resident(model, minimum_free_gib: float = 4.0) -> None:
    non_cuda = sorted({
        parameter.device.type
        for parameter in model.parameters()
        if parameter.device.type != "cuda"
    })
    offload_hooks = [
        name for name, module in model.named_modules()
        if getattr(getattr(module, "_hf_hook", None), "offload", False)
    ]
    if non_cuda or offload_hooks:
        raise RuntimeError(
            "The checkpoint did not fit on the GPU and accelerate offloaded "
            f"part of it (devices={non_cuda}, offload_hooks={len(offload_hooks)}). "
            "Restart the runtime to release stale VRAM, then rerun from the top."
        )
    free_gib = torch.cuda.mem_get_info()[0] / 1024**3
    if free_gib < minimum_free_gib:
        raise RuntimeError(
            f"Only {free_gib:.1f} GiB VRAM is free after the load; the KV "
            "cache and generation workspaces need headroom. Restart the "
            "runtime and rerun from the top."
        )
    print(f"Model fully resident on GPU; {free_gib:.1f} GiB VRAM free.")

release_stale_gpu_state()

hf_token = userdata.get("HF_TOKEN") if userdata is not None else os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN to Colab Secrets before continuing.")
login(token=hf_token, add_to_git_credential=False)
HF_USERNAME = whoami()["name"]


def require_private_repo(repo_id: str, repo_type: str = "model") -> None:
    # Refuse to publish into a Hub repo that already exists and is public.
    # private=True on create_repo, push_to_hub and hub_private_repo applies
    # only when the repo is created; an existing public repo stays public
    # and every later push lands in the open.
    from huggingface_hub import HfApi

    api = HfApi(token=hf_token)
    if not api.repo_exists(repo_id, repo_type=repo_type):
        return
    if not api.repo_info(repo_id, repo_type=repo_type).private:
        raise RuntimeError(
            f"{repo_type} repo {repo_id} exists and is public. Make it private first with "
            f"HfApi(token=hf_token).update_repo_settings(repo_id={repo_id!r}, repo_type={repo_type!r}, "
            "private=True), or publish under a new id."
        )

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"

observed_pins = {name: package_version(name) for name in COMPATIBILITY_PINS}
pin_mismatches = {
    name: {"expected": expected, "observed": observed_pins[name]}
    for name, expected in COMPATIBILITY_PINS.items()
    if observed_pins[name] != expected
}
if pin_mismatches:
    raise RuntimeError(
        "The runtime does not match the reviewed compatibility set. "
        f"Rerun the install cell with FORCE_INSTALL=True: {pin_mismatches}"
    )

RUN_ROOT = Path("/content/qwen38_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
runtime_manifest = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_gib": round(gpu_total_gib, 2),
    "packages": {
        name: package_version(name)
        for name in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets"]
    },
    "git_revisions": GIT_REVISIONS,
    "compatibility_pins": COMPATIBILITY_PINS,
}
(RUN_ROOT / "runtime_manifest.json").write_text(json.dumps(runtime_manifest, indent=2))
print(json.dumps(runtime_manifest, indent=2))
print(f"Authenticated as {HF_USERNAME}")

## Deployment tool surface

In [ ]:
# Bumped from v1 when the `shell` description stopped carrying pilot status
# text. Tool descriptions are model inputs and part of the fingerprint, so a
# wording change is a schema change.
TOOL_SCHEMA_VERSION = "qwen38-six-tools-v3"

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files below a repository-relative directory.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a UTF-8 repository file with bounded output.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search repository text using a regular expression.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_patch",
            "description": "Apply a unified diff to files inside the repository.",
            "parameters": {
                "type": "object",
                "properties": {"patch": {"type": "string"}},
                "required": ["patch"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run an allow-listed repository test profile.",
            "parameters": {
                "type": "object",
                "properties": {"profile": {"type": "string", "enum": ["unit"]}},
                "required": ["profile"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "shell",
            "description": "Run one bash command in the repository; output is bounded and a non-zero exit code is reported.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
                "additionalProperties": False,
            },
        },
    },
]

def _without_arrow_nulls(value):
    """Remove null struct fields inserted by a Datasets/Arrow round trip."""
    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            normalized = _without_arrow_nulls(item)
            if normalized is not None:
                cleaned[key] = normalized
        return cleaned
    if isinstance(value, list):
        return [_without_arrow_nulls(item) for item in value]
    return value

def unify_columns(rows: list[dict]) -> list[dict]:
    """Give every row every key that any row in the list carries.

    ``Dataset.from_list`` names its columns from the first row alone, so a
    key that row happens not to carry is dropped from the whole table: a
    corpus whose bootstrap rows predate ``lane`` silently loses the lane of
    every public row after them, and non-agentic rows are then read as
    agentic. Filling the gaps with None keeps each row's own value and
    leaves the absent ones null, which is what the readers already expect.
    """
    columns = sorted({key for row in rows for key in row})
    return [{column: row.get(column) for column in columns} for row in rows]

def canonical_tool_schema(tools: list[dict]) -> str:
    """Return a stable semantic fingerprint while retaining tool order."""
    return json.dumps(
        _without_arrow_nulls(tools),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

TOOL_SCHEMA_JSON = canonical_tool_schema(TOOLS)

def rendered_tool_schema(rendered_prompt: str) -> str:
    """Extract and canonicalise JSON tool declarations from a Qwen prompt."""
    start_tag = "<tools>"
    end_tag = "</tools>"
    if start_tag not in rendered_prompt or end_tag not in rendered_prompt:
        raise ValueError("Rendered prompt does not contain a <tools> block.")
    payload = rendered_prompt.split(start_tag, 1)[1].split(end_tag, 1)[0]
    try:
        rendered_tools = [
            json.loads(line)
            for line in payload.splitlines()
            if line.strip()
        ]
    except json.JSONDecodeError as exc:
        raise ValueError("Rendered <tools> block is not newline-delimited JSON.") from exc
    return canonical_tool_schema(rendered_tools)

def canonical_to_qwen(messages: list[dict]) -> list[dict]:
    """Merge the leading policy messages into one system message.

    The Qwen3.8 template accepts `developer` natively and merges a run of
    leading system/developer messages itself. This fold is therefore not a
    compatibility shim: it exists so training and deployment both hand the
    template one deterministically joined policy message.
    """
    converted = []
    pending_system = []
    for stored_message in messages:
        message = _without_arrow_nulls(stored_message)
        role = message["role"]
        if role in {"system", "developer"} and not converted:
            pending_system.append(str(message.get("content", "")))
            continue
        if pending_system:
            converted.append({"role": "system", "content": "\n\n".join(pending_system)})
            pending_system = []
        converted.append(message)
    if pending_system:
        converted.append({"role": "system", "content": "\n\n".join(pending_system)})
    return converted

def text_tokenizer_of(tokenizer):
    """The text tokenizer behind a multimodal processor, or the tokenizer itself.

    FastModel hands back a processor for this vision-capable checkpoint. It
    renders chat templates and decodes, but a bare positional string is read
    as an image and token-level attributes (eos, added tokens) live one
    level down. Reach through for those, and pass text= otherwise.
    """
    return getattr(tokenizer, "tokenizer", tokenizer)


def render_chat(messages: list[dict], *, add_generation_prompt: bool, reasoning_effort: str = "medium") -> str:
    return text_tokenizer_of(tokenizer).apply_chat_template(
        canonical_to_qwen(messages),
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=True,
        reasoning_effort=reasoning_effort,
        preserve_thinking=True,
    )

## Run configuration

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset, load_dataset
from trl import SFTConfig, SFTTrainer

MODEL_ID = "unsloth/Qwen3.8-27B"
MODEL_REVISION = "main"  # resolved to a commit below and recorded in the manifest
# Rank 16 left 116M trainable parameters against 27B frozen, and
# the 5,729-row run ended with its training loss above its
# validation loss: the adapter could not hold what the corpus
# had. Alpha tracks the rank so the update scaling (alpha / r)
# stays where it was and only capacity changes.
#
# 32 rather than 64 because the card decides it. That run peaked
# at 88.5 GiB of the roughly 89.4 GiB the G4 reports
# (docs/model-and-hardware.md), and each doubling of the rank
# costs about 0.65 GiB of weight, gradient and 8-bit optimiser
# state: 32 fits in that 0.9 GiB, 64 does not. Going further
# means buying room first, and the row windows are already the
# shorter end of what these trajectories need, so the sequence
# length is the wrong place to buy it from.
LORA_RANK = 32
LORA_ALPHA = 2 * LORA_RANK
DATASET_ID = f"{HF_USERNAME}/qwen38-code-native-sft-v0"
DATASET_REVISION = "main"  # the dataset notebook 02 pushed; pin a commit to repeat a run exactly
OUTPUT_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-sft-lora"
# Notebook 04 starts from the merged SFT weights, so its KL
# reference is the SFT policy rather than the base, and a DPO
# adapter trained on them is loadable elsewhere only if they are
# on the Hub. Published at the end of training, about 55 GB, with
# the repo history squashed so only the latest merge is stored.
MERGED_MODEL_ID = f"{HF_USERNAME}/qwen38-27b-code-sft-merged"
# Notebook 04's adapter is trained on the merged weights above. A
# new SFT run replaces them, so that adapter stops being the latest
# finished stage: its completion marker is removed when training
# starts, and notebook 07 gates this adapter until 04 reruns.
DPO_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-dpo-lora"
MAX_SEQ_LENGTH = 8_192       # public rows are windowed to fit this; the 4k run measured the headroom
# One pass over whatever the dataset holds; the trainer counts
# the updates. A positive MAX_STEPS would override the epochs.
# The 3,207-row run measured the second epoch: validation sat at
# 0.240 from the end of the first to the end of the second, so
# those hours now buy fresh rows instead of a repeat.
NUM_TRAIN_EPOCHS = 1
MAX_STEPS = -1
# True trains two local smoke steps on the fixture and publishes nothing.
DEMO_MODE = False
RUN_TRAINING = True
PUSH_ADAPTER = True
PUSH_MERGED_SFT = True
# docs/training-plan.md, Stage 1: 2e-5 is the main-run default
# and 5e-5, 1e-4 the sweep points. 1e-4 was for the 204-row
# bootstrap, where a rank-16 adapter had a few dozen steps to
# move at all. The corpus notebook 02 now builds is thousands
# of rows and hundreds of steps per epoch, so this drops to the
# middle of the band.
LEARNING_RATE = 5e-5
# Every step in demo mode so the smoke exercises save and eval.
# In a real run each eval reads the whole held-out split and each
# save pushes the adapter to the Hub, so the cadence is set
# against the step count: every ten steps costs more in eval and
# upload than in training once a run is hundreds of steps long.
EVAL_EVERY_STEPS = 1 if DEMO_MODE else 50
SAVE_EVERY_STEPS = 1 if DEMO_MODE else 50
# Every eval reads the whole held-out split at batch size one,
# so its cost grows with the corpus while its job, drawing a
# loss curve, does not. A fixed sample keeps the run's wall
# clock tied to the training rows: at the step and forward-pass
# costs the 3,207-row run measured, a capped run lands at the
# same four and a half hours, where reading the whole 640-row
# split every time would add well over an hour. The sample is
# seeded, so the curve is comparable between runs.
EVAL_ROW_CAP = 256

if DEMO_MODE:
    # A smoke run: two local steps on the fixture, nothing published.
    MAX_STEPS, PUSH_ADAPTER, PUSH_MERGED_SFT = 2, False, False
# The trainer creates the Hub repo when it is built, so an
# existing public repo is caught here, before that happens.
if PUSH_ADAPTER:
    require_private_repo(OUTPUT_ADAPTER_ID)
if PUSH_MERGED_SFT:
    require_private_repo(MERGED_MODEL_ID)

run_manifest = {
    "stage": "sft",
    "objective": "agentic-coding",
    "general_retention_share": 0.0,
    "model_id": MODEL_ID,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "merged_model_id": MERGED_MODEL_ID,
    "max_seq_length": MAX_SEQ_LENGTH,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": MAX_STEPS,
    "learning_rate": LEARNING_RATE,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "gradient_accumulation_steps": 8,
    "optimizer": "adamw_8bit",
    "eval_every_steps": EVAL_EVERY_STEPS,
    "eval_row_cap": EVAL_ROW_CAP,
    "save_every_steps": SAVE_EVERY_STEPS,
    "demo_mode": DEMO_MODE,
    "tool_schema_version": TOOL_SCHEMA_VERSION,
    "harness_version": "pilot-local-v1",
    "run_training": RUN_TRAINING,
}
print(json.dumps(run_manifest, indent=2))

## Load the model and discover supported LoRA targets

In [ ]:
require_free_vram(60.0)
from huggingface_hub import HfApi

# The base repository is mutable too. Resolve it once, load that
# commit, and record it: a run is otherwise unreproducible, and
# adapter state trained against one base could resume against
# another without anything noticing.
MODEL_COMMIT = HfApi(token=hf_token).model_info(MODEL_ID, revision=MODEL_REVISION).sha
run_manifest["model_commit"] = MODEL_COMMIT
print(f"{MODEL_ID}@{MODEL_REVISION} is {MODEL_COMMIT}")
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    revision=MODEL_COMMIT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    full_finetuning=False,
    token=hf_token,
)
assert_model_fully_resident(model)

from collections import Counter

# Qwen3.8 repeats three Gated DeltaNet layers per full-attention
# layer, and the DeltaNet projections are named in_proj_qkv,
# in_proj_z, in_proj_a and in_proj_b. None of those matched the
# earlier suffix list, so 48 of the 64 layers had no adapter on
# their attention path at all. Discover the set, then refuse to
# train if it is not the reviewed one.
REVIEWED_TARGET_SUFFIXES = {
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj_qkv", "in_proj_z", "in_proj_a", "in_proj_b", "out_proj",
    "gate_proj", "up_proj", "down_proj",
}
# The vision tower is frozen by project policy. The MTP head and
# lm_head are excluded because nothing here trains a multi-token
# objective and the merged checkpoint should keep its original
# output layer.
EXCLUDED_MODULE_MARKERS = ("visual", "vision", "image", "mtp.", "lm_head")

def is_excluded_module(name: str) -> bool:
    lowered = name.lower()
    return any(marker in lowered for marker in EXCLUDED_MODULE_MARKERS)

language_linear_names = [
    name for name, module in model.named_modules()
    if isinstance(module, torch.nn.Linear) and not is_excluded_module(name)
]
discovered_suffixes = {name.rsplit(".", 1)[-1] for name in language_linear_names}
expected_module_counts = Counter(name.rsplit(".", 1)[-1] for name in language_linear_names)
missing_suffixes = sorted(REVIEWED_TARGET_SUFFIXES - discovered_suffixes)
unexpected_suffixes = sorted(discovered_suffixes - REVIEWED_TARGET_SUFFIXES)
if missing_suffixes:
    raise RuntimeError(
        f"Reviewed LoRA targets are absent from the loaded model: {missing_suffixes}. "
        "The architecture or the loader changed; re-derive the target list before training."
    )
if unexpected_suffixes:
    raise RuntimeError(
        f"The model exposes language linear modules this suite has not reviewed: {unexpected_suffixes}. "
        "Decide explicitly whether they belong in the adapter, then update REVIEWED_TARGET_SUFFIXES."
    )
target_modules = sorted(discovered_suffixes)
print(json.dumps({
    "lora_targets": target_modules,
    "language_linear_modules": dict(sorted(expected_module_counts.items())),
}, indent=2))

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,  # text-only specialisation; the reviewed list below decides the rest
    r=LORA_RANK,
    target_modules=target_modules,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# PEFT matches target_modules by name suffix, so the MTP head's
# own q_proj/o_proj would otherwise receive adapters that no loss
# ever trains. Freeze everything outside the language decoder.
for name, parameter in model.named_parameters():
    if is_excluded_module(name):
        parameter.requires_grad_(False)
trainable_outside_decoder = [
    name for name, parameter in model.named_parameters()
    if parameter.requires_grad and is_excluded_module(name)
]
assert not trainable_outside_decoder, trainable_outside_decoder[:20]

# Prove the adapter actually reaches every module discovery found,
# rather than trusting that suffix matching did what was intended.
adapted_counts = Counter(
    name.rsplit(".", 1)[-1]
    for name, module in model.named_modules()
    if not is_excluded_module(name) and len(getattr(module, "lora_A", {}) or {})
)
if adapted_counts != expected_module_counts:
    raise RuntimeError(
        "LoRA coverage does not match module discovery. "
        f"expected={dict(sorted(expected_module_counts.items()))} "
        f"adapted={dict(sorted(adapted_counts.items()))}"
    )
print(json.dumps({"adapted_modules": dict(sorted(adapted_counts.items()))}, indent=2))
model.print_trainable_parameters()

## Load native-schema data and render the exact deployment template

In [ ]:
def demo_rows():
    return [
        {
            "repo_family": "fixture/clamp",
            "tool_schema_version": TOOL_SCHEMA_VERSION,
            "tool_schema_json": TOOL_SCHEMA_JSON,
            "tools": TOOLS,
            "messages": [
                {"role": "developer", "content": "Fix the bug, run tests, and keep the change minimal."},
                {"role": "user", "content": "clamp() returns values outside its bounds."},
                {"role": "assistant", "content": "", "tool_calls": [{
                    "type": "function",
                    "function": {"name": "read_file", "arguments": {"path": "src/clamp.py"}},
                }]},
                {"role": "tool", "name": "read_file", "content": "def clamp(x, low, high):\n    return x\n"},
                {"role": "assistant", "content": "", "tool_calls": [{
                    "type": "function",
                    "function": {"name": "apply_patch", "arguments": {
                        "patch": "--- a/src/clamp.py\n+++ b/src/clamp.py\n@@ -1,2 +1,2 @@\n def clamp(x, low, high):\n-    return x\n+    return max(low, min(high, x))\n"
                    }},
                }]},
                {"role": "tool", "name": "apply_patch", "content": "Done!"},
                {"role": "assistant", "content": "Implemented the bounded clamp and kept the patch focused."},
            ],
        },
        {
            "repo_family": "fixture/parser",
            "tool_schema_version": TOOL_SCHEMA_VERSION,
            "tool_schema_json": TOOL_SCHEMA_JSON,
            "tools": TOOLS,
            "messages": [
                {"role": "developer", "content": "Investigate first, then make the smallest correct edit."},
                {"role": "user", "content": "Return an empty list for an empty CSV field."},
                {"role": "assistant", "content": "I will inspect the parser and its tests before editing."},
            ],
        },
    ]

USE_DEMO_DATA = DEMO_MODE
DATASET_COMMIT = None
if USE_DEMO_DATA:
    raw = Dataset.from_list(demo_rows())
    split = raw.train_test_split(test_size=0.5, seed=3407)
    train_raw, eval_raw = split["train"], split["test"]
    print("Using synthetic plumbing data; this is not a capability run.")
else:
    from huggingface_hub import HfApi

    # "main" moves whenever notebook 02 republishes. Resolve it
    # once and load that commit, so what is loaded, what the
    # manifest records and what keys the checkpoints are the
    # same corpus. Row counts alone would not tell two corpora
    # apart: the source caps are hit exactly, so a changed
    # converter republishes the same number of different rows.
    DATASET_COMMIT = HfApi(token=hf_token).dataset_info(
        DATASET_ID, revision=DATASET_REVISION
    ).sha
    run_manifest["dataset_commit"] = DATASET_COMMIT
    print(f"{DATASET_ID}@{DATASET_REVISION} is {DATASET_COMMIT}")
    loaded = load_dataset(DATASET_ID, revision=DATASET_COMMIT, token=hf_token)
    missing_splits = {"train", "validation"} - set(loaded)
    if missing_splits:
        raise ValueError(
            f"Dataset is missing required repository-family splits: {sorted(missing_splits)}. "
            "Rebuild it with notebook 02."
        )
    if len(loaded["train"]) == 0 or len(loaded["validation"]) == 0:
        raise ValueError("Both train and validation splits must contain at least one repository family.")
    fixture_rows = sum(
        str(row_id).startswith("fixture/")
        for split in ("train", "validation")
        for row_id in (loaded[split]["id"] if "id" in loaded[split].column_names else [])
    )
    if fixture_rows:
        raise ValueError(
            f"{DATASET_ID}@{DATASET_REVISION} holds notebook 02's format fixture ({fixture_rows} rows), "
            "not a corpus. Rerun notebook 02 with DEMO_MODE=False and push again."
        )
    train_raw = loaded["train"]
    eval_raw = loaded["validation"]

def render_row(row):
    if row.get("tool_schema_version") != TOOL_SCHEMA_VERSION:
        raise ValueError("Dataset schema version differs from this notebook.")
    if row.get("tool_schema_json") != TOOL_SCHEMA_JSON:
        raise ValueError("Dataset canonical tool fingerprint differs from this notebook.")
    if canonical_tool_schema(row.get("tools") or []) != TOOL_SCHEMA_JSON:
        raise ValueError("Dataset tools differ from the deployment tool surface.")
    return {
        "text": render_chat(
            row["messages"],
            add_generation_prompt=False,
            reasoning_effort=row.get("reasoning_effort") or "medium",
        )
    }

eval_rows_available = len(eval_raw)
if EVAL_ROW_CAP and eval_rows_available > EVAL_ROW_CAP:
    # Taking the first rows of a shuffle can drop a lane the
    # split went to the trouble of holding out. Interleaving the
    # lanes first keeps each one in the sample, in the shuffled
    # order, and makes the sample as even as the split allows.
    shuffled = eval_raw.shuffle(seed=3407)
    lane_values = (
        shuffled["lane"] if "lane" in shuffled.column_names
        else [None] * len(shuffled)
    )
    by_lane = {}
    for position, lane in enumerate(lane_values):
        by_lane.setdefault(lane or "agentic", []).append(position)
    groups = [by_lane[lane] for lane in sorted(by_lane)]
    interleaved = [
        group[depth]
        for depth in range(max(len(group) for group in groups))
        for group in groups
        if depth < len(group)
    ]
    eval_raw = shuffled.select(interleaved[:EVAL_ROW_CAP])
train_dataset = train_raw.map(render_row)
eval_dataset = eval_raw.map(render_row)
print(json.dumps({
    "train_rows": len(train_dataset),
    "eval_rows": len(eval_dataset),
    "eval_rows_available": eval_rows_available,
}, indent=2))
print(train_dataset[0]["text"][:4000])

## Build the assistant-only trainer and inspect its labels

In [ ]:
import hashlib

# A rerun in a runtime that still holds the last run's
# checkpoints resumes them: same directory, and the resume is
# unconditional. When the corpus or the schedule has changed
# since, that trains from the wrong state, or skips training
# outright because the old run went further, and then publishes
# the result under this run's manifest. The directory is keyed
# by what decides the schedule, so a changed run starts clean
# while an interrupted identical one still resumes.
SFT_RUN_KEY = hashlib.sha256(json.dumps({
    "model_commit": MODEL_COMMIT,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_COMMIT or DATASET_REVISION,
    "train_rows": len(train_dataset),
    "eval_rows": len(eval_dataset),
    "max_seq_length": MAX_SEQ_LENGTH,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": MAX_STEPS,
    "learning_rate": LEARNING_RATE,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "gradient_accumulation_steps": 8,
    "seed": 3407,
}, sort_keys=True).encode()).hexdigest()[:12]
SFT_RUN_DIR = RUN_ROOT / "sft" / SFT_RUN_KEY
SFT_RUN_DIR.mkdir(parents=True, exist_ok=True)
run_manifest["run_key"] = SFT_RUN_KEY
print(f"checkpoints and artifacts for this configuration: {SFT_RUN_DIR}")

training_args = SFTConfig(
    output_dir=str(SFT_RUN_DIR),
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    bf16=True,
    fp16=False,
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY_STEPS,
    save_strategy="steps",
    save_steps=SAVE_EVERY_STEPS,
    save_total_limit=2,
    seed=3407,
    report_to="trackio",
    run_name="qwen38-code-sft-smoke" if USE_DEMO_DATA else "qwen38-code-sft",
    push_to_hub=PUSH_ADAPTER,
    hub_model_id=OUTPUT_ADAPTER_ID,
    hub_strategy="every_save",
    hub_private_repo=True,
)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

batch = next(iter(trainer.get_train_dataloader()))
labels = batch["labels"]
assert (labels != -100).any(), "No assistant tokens remain after response masking."
first_trainable = int((labels[0] != -100).nonzero()[0])
assert (labels[0, :first_trainable] == -100).all(), "Prompt/tool context leaked into the first response loss."

# Checking for one phrase from the demo fixture made this gate
# pass only in demo mode: with real data it failed before
# training could start. Derive the expectation from each row.
#
# The chat template wraps every observation in this tag, so the
# tag inside the loss is an observation inside the loss. Looking
# for the observation's own text instead refused real agentic
# rows: an observation prints a path, the assistant's next
# command names that path, and the path is then in the loss
# while the observation is masked exactly as it should be.
OBSERVATION_TAG = "<tool_response>"

def masking_problems(tokenized_split, source_split) -> list[str]:
    if len(tokenized_split) != len(source_split):
        return [
            f"trainer holds {len(tokenized_split)} rows but the source has "
            f"{len(source_split)}; rows cannot be compared positionally"
        ]
    problems = []
    for index, (row, source_row) in enumerate(zip(tokenized_split, source_split)):
        supervised_ids = [
            token_id for token_id, label in zip(row["input_ids"], row["labels"])
            if label != -100
        ]
        if not supervised_ids:
            problems.append(f"row {index}: no supervised tokens remain")
            continue
        supervised = tokenizer.decode(supervised_ids, skip_special_tokens=False)
        if OBSERVATION_TAG in supervised:
            problems.append(f"row {index}: tool observation leaked into the loss")
        # A right-truncated row legitimately loses its tail, so
        # only assert completeness where nothing was cut.
        complete = len(row["input_ids"]) < MAX_SEQ_LENGTH
        if not complete:
            continue
        for message in source_row["messages"]:
            content = (message.get("content") or "").strip()
            if message["role"] == "assistant" and content and content not in supervised:
                problems.append(f"row {index}: assistant content is masked out of the loss")
    return problems

masking_failures = (
    masking_problems(trainer.train_dataset, train_dataset)
    + masking_problems(trainer.eval_dataset, eval_dataset)
)
if masking_failures:
    raise RuntimeError(f"Assistant-only masking is wrong (first 10): {masking_failures[:10]}")

first_supervised = tokenizer.decode(
    [
        token_id for token_id, label in zip(
            trainer.train_dataset[0]["input_ids"], trainer.train_dataset[0]["labels"]
        )
        if label != -100
    ],
    skip_special_tokens=False,
)
print({
    "batch_shape": tuple(labels.shape),
    "first_trained_token": first_trainable,
    "checked_rows": len(trainer.train_dataset) + len(trainer.eval_dataset),
    "supervision_preview": first_supervised[:2000],
})

## Train, resume, and publish the adapter

In [ ]:
# checkpoint-10 sorts before checkpoint-9 lexicographically, so a
# plain sort silently resumes from a stale step.
def latest_checkpoint(directory):
    numbered = [
        (int(path.name.rsplit("-", 1)[-1]), path)
        for path in directory.glob("checkpoint-*")
        if path.is_dir() and path.name.rsplit("-", 1)[-1].isdigit()
    ]
    return max(numbered)[1] if numbered else None

if RUN_TRAINING:
    resume_from = latest_checkpoint(SFT_RUN_DIR)
    torch.cuda.reset_peak_memory_stats()
    start_reserved_gib = torch.cuda.memory_reserved() / 1024**3
    if PUSH_ADAPTER:
        # An earlier run's completion marker must not survive into
        # this run's intermediate pushes, or notebook 07 would take
        # a half-trained adapter for a finished one. Removed here,
        # once training is certain to start, so a dry run or a
        # failure before this point leaves a valid adapter alone.
        from huggingface_hub import HfApi

        hub = HfApi(token=hf_token)
        if hub.repo_exists(OUTPUT_ADAPTER_ID) and hub.file_exists(OUTPUT_ADAPTER_ID, "run_manifest.json"):
            hub.delete_file(
                "run_manifest.json", OUTPUT_ADAPTER_ID,
                commit_message="training started: completion marker removed",
            )
        # The DPO adapter descends from the merged weights this run
        # replaces, so it is no longer the latest finished stage.
        if hub.repo_exists(DPO_ADAPTER_ID) and hub.file_exists(DPO_ADAPTER_ID, "run_manifest.json"):
            hub.delete_file(
                "run_manifest.json", DPO_ADAPTER_ID,
                commit_message="SFT lineage replaced: completion marker removed",
            )
    result = trainer.train(resume_from_checkpoint=str(resume_from) if resume_from else None)
    peak_reserved_gib = torch.cuda.max_memory_reserved() / 1024**3
    run_manifest["train_runtime_seconds"] = result.metrics.get("train_runtime")
    run_manifest["peak_reserved_gib"] = round(peak_reserved_gib, 3)
    run_manifest["training_memory_delta_gib"] = round(peak_reserved_gib - start_reserved_gib, 3)
    trainer.save_model(str(SFT_RUN_DIR / "final_adapter"))
    tokenizer.save_pretrained(str(SFT_RUN_DIR / "final_adapter"))
    (SFT_RUN_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
    if PUSH_ADAPTER:
        from huggingface_hub import HfApi

        trainer.push_to_hub(commit_message="SFT adapter with native six-tool schema")
        # A completion marker. The trainer created the repo before
        # training, so its existence proves nothing; notebook 07
        # gates an adapter only once this file is on the Hub.
        HfApi(token=hf_token).upload_file(
            path_or_fileobj=str(SFT_RUN_DIR / "run_manifest.json"),
            path_in_repo="run_manifest.json",
            repo_id=OUTPUT_ADAPTER_ID,
            commit_message="run manifest: training completed",
        )
    if PUSH_MERGED_SFT:
        # The merged weights notebook 04 starts from. The marker
        # goes up after the weights, and the history is squashed
        # so the repo holds one copy (about 55 GB), not one per
        # run. That discards the parent of any DPO adapter
        # trained on the previous merge, which is why this run
        # removed that adapter's completion marker when training
        # started: nothing downstream treats it as current, and
        # its manifest keeps the commit it was trained on.
        from huggingface_hub import HfApi

        hub = HfApi(token=hf_token)
        hub.create_repo(MERGED_MODEL_ID, repo_type="model", private=True, exist_ok=True)
        if hub.file_exists(MERGED_MODEL_ID, "run_manifest.json"):
            hub.delete_file(
                "run_manifest.json", MERGED_MODEL_ID,
                commit_message="merge started: completion marker removed",
            )
        model.push_to_hub_merged(MERGED_MODEL_ID, tokenizer, save_method="merged_16bit", token=hf_token)
        hub.upload_file(
            path_or_fileobj=str(SFT_RUN_DIR / "run_manifest.json"),
            path_in_repo="run_manifest.json",
            repo_id=MERGED_MODEL_ID,
            commit_message="run manifest: merge completed",
        )
        hub.super_squash_history(MERGED_MODEL_ID, commit_message="keep only the latest merge")
        print(f"Published the merged SFT checkpoint to {MERGED_MODEL_ID}.")
    print(result.metrics)
    print({
        "peak_reserved_gib": round(peak_reserved_gib, 3),
        "training_memory_delta_gib": round(peak_reserved_gib - start_reserved_gib, 3),
    })
else:
    print("Dry run complete. Set RUN_TRAINING=True only after inspecting labels and memory.")

## Gate to notebook 04

Training loss is diagnostic, not success. Accept this adapter only
if native tool syntax, held-out patch correctness, non-regression,
and sentinel long-horizon outcomes beat the frozen baseline.
Record the exact adapter commit SHA before preference tuning.